In [1]:
import pandas as pd
import numpy as np

In [2]:
df_watch_history = pd.read_csv(r"C:\Users\Bhagyashri\Downloads\ff5a102e4361911d42cef9e21cda23d0\watch_history.csv")

### **Calculation of Total Watch Hours**

In [3]:
watch_hours = df_watch_history['watch_duration_min'].sum()/60

In [4]:
print(f"Total watch hours: {watch_hours:,.2f} Hours")

Total watch hours: 3,333,467.73 Hours


In [5]:
df_subscribers = pd.read_csv(r"C:\Users\Bhagyashri\Downloads\ff5a102e4361911d42cef9e21cda23d0\subscribers.csv")

### **Calculation of Subscribers Active Rate in %**

In [6]:
active_rate = df_subscribers['is_active'].mean()*100

In [7]:
print(f"Total Active Rate of subscribers: {active_rate:,.2f} %")

Total Active Rate of subscribers: 74.66 %


### **Calculation of Subscribers Churn Rate in %**

In [8]:
churn_rate = 100 - active_rate

In [9]:
print(f"Total Churn Rate of subscribers: {churn_rate:,.2f} %")

Total Churn Rate of subscribers: 25.34 %


### **Calculation of Avg Completion Rate in %**

In [10]:
Avg_Completion_Rate = df_watch_history['completion_pct'].mean()

In [11]:
print(f"Total Average Completion Rate of Subscribers: {Avg_Completion_Rate:,.2f} %")

Total Average Completion Rate of Subscribers: 65.31 %


### **Calculation of Monthly Recurring Revenue**

In [12]:
active_subscribers = df_subscribers[df_subscribers['is_active']== True]

In [13]:
print(f"Total Active Subscribers: {len(active_subscribers)}")

Total Active Subscribers: 11199


In [14]:
Monthly_Recurring_Revenue = active_subscribers['monthly_price_usd'].sum()

In [15]:
print(f"Monthly Recurring Revenue (MRR): ${Monthly_Recurring_Revenue:,.2F}")

Monthly Recurring Revenue (MRR): $175,868.51


### **Calculation of Average Revenue Per User**

In [19]:
Average_Revenue_Per_User = Monthly_Recurring_Revenue / len(active_subscribers)

In [22]:
print(f"Average Revenue Per Active User(ARPU): ${Average_Revenue_Per_User:,.2F}")

Average Revenue Per Active User(ARPU): $15.70


### **Calculation of Avg Watch Time / Subscriber**

In [27]:
avg_watch_time_per_subscriber = watch_hours / len(active_subscribers)

In [30]:
print(f"Avgrage Watch Time Per Subscriber: {avg_watch_time_per_subscriber:,.2f} Hour")

Avgrage Watch Time Per Subscriber: 297.66 Hour


### **Calculation of Watchlist Conversion**

In [32]:
df_watchlist = pd.read_csv(r"C:\Users\Bhagyashri\Downloads\ff5a102e4361911d42cef9e21cda23d0\watchlist.csv")

In [33]:
total_watchlist_entries = df_watchlist['watchlist_id'].count()

In [35]:
print(f"Total of Watchlist Entries: {total_watchlist_entries:}")

Total of Watchlist Entries: 65000


In [38]:
watched = df_watchlist[df_watchlist['watched']== True]

In [39]:
print(f"Total of Watched: {len(watched)}")

Total of Watched: 30048


In [43]:
Watchlist_Conversion = (len(watched) / total_watchlist_entries) * 100

In [44]:
print(f"Watchlist Conversion: {Watchlist_Conversion:,.2f}%")

Watchlist Conversion: 46.23%


### **Calculation of Hit Concentration**

In [45]:
df_titles = pd.read_csv(r"C:\Users\Bhagyashri\Downloads\ff5a102e4361911d42cef9e21cda23d0\titles.csv")

### Step 1: Calculate total plays (views) per title and sort them in descending order

In [50]:
title_plays = df_watch_history.groupby('title_id')['watch_duration_min'].count().sort_values(ascending=False)

### Step 2: Determine the count representing the top 10% of unique titles

In [48]:
top_10_percent_count = int(len(title_plays)*0.10)

### Step 3: Calculate the total plays for top 10% titles and compute the percentage against total plays

In [51]:
top_plays = title_plays.iloc[:top_10_percent_count].sum()
total_plays = title_plays.sum()

In [52]:
hit_concentration = (top_plays / total_plays) * 100

### Step 4: Display the result

In [53]:
print(f"9. Hit Concentration (Top 10% titles): {hit_concentration:.2f}%")

9. Hit Concentration (Top 10% titles): 30.99%


### **Calculation of Originals Share of Hours**

In [54]:
merge_dataset = pd.merge(df_titles,df_watch_history, on ='title_id',how='inner')

### Step 2 Total watch time of Original Content in minutes

In [61]:
original_watch_minutes = merge_dataset[merge_dataset['license_type'] == 'Original']['watch_duration_min'].sum()

### Step 2 Total watch time of all Content in minutes

In [62]:
total_watch_minutes = merge_dataset['watch_duration_min'].sum()

### Step 3 KPI 10: Originals Share of Hours (%)

In [63]:
originals_share = (original_watch_minutes / total_watch_minutes) * 100

In [64]:
print(f"10. Originals Share of Hours: {originals_share:.2f}%")

10. Originals Share of Hours: 27.17%


# ==========================================
# Bonus Challenge: Cohort Retention Rate
# ==========================================

### Step 1: Ensure signup_date is in datetime format and extract Signup Month (Cohort)

In [65]:
df_subscribers['signup_date'] = pd.to_datetime(df_subscribers['signup_date'])

In [66]:
df_subscribers['cohort_month'] = df_subscribers['signup_date'].dt.to_period('M')

### Step 2: Prepare Watch History & Merge

1. Convert watch_date column to datetime format

In [67]:
df_watch_history['watch_date'] = pd.to_datetime(df_watch_history['watch_date'])

In [69]:
df_watch_history['activity_month'] = df_watch_history['watch_date'].dt.to_period('M')

In [71]:
cohort_data = pd.merge(df_subscribers,df_watch_history, on ='subscriber_id', how='inner')

### Step 3: Calculate Month Difference (Cohort Index)

Calculate the difference in months between the activity month and the signup (cohort) month

In [72]:
cohort_data['cohort_index'] = (cohort_data['activity_month'] - cohort_data['cohort_month']).apply(lambda x: x.n)

### Step 4: Create Pivot Table & Calculate Retention Rate (%)

1. Calculate the total unique signups for each cohort month

In [76]:
cohort_sizes = df_subscribers.groupby('cohort_month')['subscriber_id'].nunique()

2. Create a pivot table to count unique active subscribers per cohort and month index

In [77]:
cohort_pivot = cohort_data.pivot_table(index='cohort_month', 
                                         columns='cohort_index', 
                                         values='subscriber_id', 
                                         aggfunc='nunique')

3. Calculate retention percentage (%) by dividing active users by actual signup cohort sizes

In [78]:
retention_matrix = cohort_pivot.divide(cohort_sizes, axis=0) * 100

4. Fill NaN values with 0.00 and filter for the last 6 cohorts and Month 0 to Month 6

In [79]:
clean_matrix = retention_matrix.fillna(0).tail(6)[[0, 1, 2, 3, 4, 5, 6]]

5. Display the clean retention rate matrix

In [80]:
print("--- Clean Cohort Retention Rate (%) ---")
print(clean_matrix.round(2))

--- Clean Cohort Retention Rate (%) ---
cohort_index       0       1       2       3       4      5    6
cohort_month                                                    
2025-12        13.36   61.75   74.65   85.25   94.47  99.54  0.0
2026-01        25.86   68.97   91.38   98.71  100.00   0.00  0.0
2026-02        33.33   87.22   98.33  100.00    0.00   0.00  0.0
2026-03        50.93   97.20  100.00    0.00    0.00   0.00  0.0
2026-04        64.76  100.00    0.00    0.00    0.00   0.00  0.0
2026-05       100.00    0.00    0.00    0.00    0.00   0.00  0.0


# 📊 Cohort Retention Rate Analysis (Key Insights)

* **2026-01 Cohort:** 
  * Started at **25.86%** active engagement in Month 0.
  * Showed steady growth in engagement, reaching **100%** active retention by Month 4.
* **2026-02 Cohort:** 
  * Retained **87.22%** users in Month 1 and reached **100%** retention in Month 3.
* **2026-05 Cohort:** 
  * Achieved **100%** immediate user engagement in Month 0.

**Conclusion:** 
The platform shows strong long-term user retention, with recent cohorts demonstrating faster initial engagement post-signup.